# SVGP Digital Twin — Proof of Concept

**Sprint 1 Deliverable** — Side-by-side comparison of `BayesianDTModel` (ExactGP) and `SVGPDigitalTwin` (Sparse Variational GP).

## What this notebook demonstrates

| | Bayesian (ExactGP) | SVGP |
|---|---|---|
| Kernel | ScaleKernel(RBF) + ConstantMean | Same |
| Training | Exact MLL, all data in memory | Variational ELBO, mini-batches |
| Complexity | O(n³) — fails at >10K pts | O(n·m²) — scales to 100K+ pts |
| Save/Load | Pickle (security risk) | torch.save (state_dict) |
| Interface | Direct BayesianDigitalTwin API | DTModel ABC |

### Data pipeline
1. **Generate UE tracks** — Gauss-Markov mobility model over a 3-cell deployment
2. **Synthesize RSRP** — log-distance path loss + lognormal shadow fading
3. **Feature engineering** — log_distance, relative_bearing (same as production pipeline)
4. **Train & predict** — both models, same data, same splits
5. **Compare** — MAE, prediction uncertainty, training time

### Key insight: accuracy vs. scale
SVGP does **not** automatically beat Bayesian on accuracy — it's an approximation.
On small data (< 1K points/cell), ExactGP is marginally more accurate.
On real deployment data (1K–100K+ points/cell), **Bayesian can't run** (O(n³) OOM),
while SVGP handles it trivially. That's where SVGP wins decisively.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))


In [2]:
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from radp.digital_twin.utils import constants as c
from radp.digital_twin.utils.gis_tools import GISTools
from radp.utility.simulation_utils import seed_everything

from radp.digital_twin.rf.base_model import DTModel
from radp.digital_twin.rf.svgp.svgp_engine import SVGPDigitalTwin, SVGPTrainConfig

try:
    from radp.digital_twin.rf.bayesian.bayesian_engine import BayesianDigitalTwin
except ImportError:
    BayesianDigitalTwin = None

# Set to True only for small data. ExactGP scales cubically and can be very slow.
RUN_BAYESIAN = False

# Data paths
DATA_DIR = REPO_ROOT / "tmp" / "data" / "Million"

seed_everything(42)
print("Imports OK  |  DATA_DIR:", DATA_DIR)


Imports OK  |  DATA_DIR: /Users/tanzimfarhan/Desktop/Maveric/maveric_cloudly/maveric/tmp/data/Million


## 1. Load topology and config

In [3]:
topology_df = pd.read_csv(DATA_DIR / "topology.csv")
config_df = pd.read_csv(DATA_DIR / "config.csv")

# Current SVGPDigitalTwin requires antenna-height columns for feature engineering.
# The Million sample data does not include them, so use standard macro defaults.
DEFAULT_HTX_M = 30.0
DEFAULT_HRX_M = 1.5

site_configs_df = topology_df.merge(
    config_df[["cell_id", "cell_el_deg"]],
    on="cell_id",
    how="left",
)
site_configs_df[c.HTX] = site_configs_df.get(c.HTX, DEFAULT_HTX_M)
site_configs_df[c.HRX] = site_configs_df.get(c.HRX, DEFAULT_HRX_M)

TOPO_COLS = [
    c.CELL_ID,
    c.CELL_AZ_DEG,
    c.CELL_LAT,
    c.CELL_LON,
    c.CELL_CARRIER_FREQ_MHZ,
    c.CELL_EL_DEG,
    c.HTX,
    c.HRX,
    "site_id",
]
site_configs_df = site_configs_df[TOPO_COLS].reset_index(drop=True)

print(f"Loaded {len(site_configs_df)} cells across {site_configs_df.site_id.nunique()} sites")
site_configs_df.head()


Loaded 20 cells across 5 sites


,cell_id,cell_az_deg,cell_lat,cell_lon,cell_carrier_freq_mhz,cell_el_deg,hTx,hRx,site_id
0,cell_1_0,356.242833,38.71404,-125.836106,1800,8.706244,30.0,1.5,Site1
1,cell_1_1,91.504593,38.71404,-125.836106,2100,8.884467,30.0,1.5,Site1
2,cell_1_2,183.894878,38.71404,-125.836106,850,11.340435,30.0,1.5,Site1
3,cell_1_3,273.902744,38.71404,-125.836106,2100,4.271576,30.0,1.5,Site1
4,cell_2_0,355.585680,41.60740,-123.573457,2100,5.361171,30.0,1.5,Site2


## 2. Load and filter UE training data

In [4]:
ue_raw = pd.read_csv(DATA_DIR / "ue_training_data.csv")

raw_rows = len(ue_raw)
raw_cells = ue_raw[c.CELL_ID].nunique()
no_signal_rows = int((ue_raw["avg_rsrp"] <= -140).sum())
valid_signal_rows = int((ue_raw["avg_rsrp"] > -140).sum())
rows_per_cell = ue_raw.groupby(c.CELL_ID).size()

print(f"Raw UE rows      : {raw_rows:,}")
print(f"Cells            : {raw_cells}")
print(f"Rows per cell    : min={rows_per_cell.min():,}, max={rows_per_cell.max():,}")
print(f"avg_rsrp <= -140 : {no_signal_rows:,}")
print(f"avg_rsrp >  -140 : {valid_signal_rows:,}")
print(f"Columns          : {list(ue_raw.columns)}")


Raw UE rows      : 1,000,000
Cells            : 20
Rows per cell    : min=50,000, max=50,000
avg_rsrp <= -140 : 800,431
avg_rsrp >  -140 : 199,569
Columns          : ['cell_id', 'avg_rsrp', 'lon', 'lat', 'cell_el_deg']


In [5]:
# Select data scope for training.
# Use "ALL" for all sites, or "Site1" ... "Site5" for a smaller run.
DEMO_SITE = "ALL"

# Keep False for full 1M-row SVGP training/evaluation.
# Set True only if you explicitly want to exclude no-signal/out-of-range rows.
FILTER_NO_SIGNAL_ROWS = False

ue_model_input = ue_raw.copy()
if FILTER_NO_SIGNAL_ROWS:
    ue_model_input = ue_model_input[ue_model_input["avg_rsrp"] > -140].copy()

if DEMO_SITE == "ALL":
    demo_cell_ids = sorted(site_configs_df[c.CELL_ID].unique().tolist())
    site_configs_demo = site_configs_df.copy()
    ue_data_df = ue_model_input.copy()
else:
    demo_cell_ids = sorted(
        site_configs_df.loc[site_configs_df.site_id == DEMO_SITE, c.CELL_ID].tolist()
    )
    site_configs_demo = site_configs_df[site_configs_df[c.CELL_ID].isin(demo_cell_ids)].copy()
    ue_data_df = ue_model_input[ue_model_input[c.CELL_ID].isin(demo_cell_ids)].copy()

print(f"Scope: {DEMO_SITE}  ->  cells: {len(demo_cell_ids)}")
print(f"Total UE rows in scope: {len(ue_data_df):,}")
print(f"No-signal filtering enabled: {FILTER_NO_SIGNAL_ROWS}")
print("\nRows per site:")
print(
    ue_data_df.merge(site_configs_df[[c.CELL_ID, "site_id"]], on=c.CELL_ID, how="left")
    .groupby("site_id")
    .size()
    .to_string()
)
print("\nRows per cell:")
print(ue_data_df.groupby(c.CELL_ID).size().describe().round(0).to_string())
print("\nRSRP summary:")
print(ue_data_df.groupby(c.CELL_ID)["avg_rsrp"].describe().round(2).to_string())


Scope: ALL  ->  cells: 20
Total UE rows in scope: 1,000,000
No-signal filtering enabled: False

Rows per site:
site_id
Site1    200000
Site2    200000
Site3    200000
Site4    200000
Site5    200000

Rows per cell:
count       20.0
mean     50000.0
std          0.0
min      50000.0
25%      50000.0
50%      50000.0
75%      50000.0
max      50000.0

RSRP summary:
            count    mean    std    min    25%    50%    75%   max
cell_id                                                           
cell_1_0  50000.0 -127.66  27.56 -140.0 -140.0 -140.0 -140.0 -44.0
cell_1_1  50000.0 -128.26  26.63 -140.0 -140.0 -140.0 -140.0 -44.0
cell_1_2  50000.0 -127.40  27.90 -140.0 -140.0 -140.0 -140.0 -44.0
cell_1_3  50000.0 -127.88  27.21 -140.0 -140.0 -140.0 -140.0 -44.0
cell_2_0  50000.0 -130.35  22.45 -140.0 -140.0 -140.0 -140.0 -44.0
cell_2_1  50000.0 -130.00  22.83 -140.0 -140.0 -140.0 -140.0 -44.0
cell_2_2  50000.0 -130.41  22.45 -140.0 -140.0 -140.0 -140.0 -44.0
cell_2_3  50000.0 -130.41  22.6

In [6]:
LAT_MIN = ue_data_df["lat"].min() - 0.02
LAT_MAX = ue_data_df["lat"].max() + 0.02
LON_MIN = ue_data_df["lon"].min() - 0.02
LON_MAX = ue_data_df["lon"].max() + 0.02

# Plot a sample only; training/evaluation below still uses the full scoped data.
PLOT_MAX_POINTS = 100_000
plot_df = ue_data_df.sample(n=min(PLOT_MAX_POINTS, len(ue_data_df)), random_state=42)

fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(plot_df["lon"], plot_df["lat"],
                c=plot_df["avg_rsrp"], cmap="RdYlGn",
                vmin=-140, vmax=-50, s=4, alpha=0.35)

colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6", "#16a085"]
for idx, (_, row) in enumerate(site_configs_demo.iterrows()):
    color = colors[idx % len(colors)]
    ax.scatter(row.cell_lon, row.cell_lat, s=120, marker="^",
               c=color, edgecolors="black", linewidths=1.0,
               zorder=10, label=row.cell_id if idx < 12 else None)

plt.colorbar(sc, ax=ax, label="avg_rsrp (dBm)")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title(f"{DEMO_SITE} - UE measurement sample ({len(plot_df):,}/{len(ue_data_df):,} rows)")
ax.legend(loc="upper right", fontsize=7, ncol=2)
plt.tight_layout(); plt.show()


## 3. Feature engineering — log_distance and relative_bearing

In [7]:
print("Engineering SVGP features with current engine helpers...")
t0 = time.time()
training_frames_all = SVGPDigitalTwin.preprocess_ue_training_data(ue_data_df, site_configs_demo)
ue_data_df = pd.concat(training_frames_all, ignore_index=True)
print(f"Done in {time.time() - t0:.1f}s")

ue_data_df[[
    c.CELL_ID,
    c.LOG_DISTANCE,
    c.RELATIVE_BEARING,
    c.ANTENNA_GAIN,
    c.CELL_EL_DEG,
    "avg_rsrp",
]].head(6)


Engineering SVGP features with current engine helpers...
Done in 1.7s


,cell_id,log_distance,relative_bearing,antenna_gain,cell_el_deg,avg_rsrp
0,cell_1_0,12.000018,36.639876,-9.074891,8.706244,-140.0
1,cell_1_0,12.500556,119.046312,-9.083139,8.706244,-140.0
2,cell_1_0,12.500020,119.069101,-9.083132,8.706244,-140.0
3,cell_1_0,12.000068,36.600322,-9.074892,8.706244,-140.0
4,cell_1_0,12.835188,33.896051,-9.086751,8.706244,-140.0
5,cell_1_0,11.999424,36.644644,-9.074878,8.706244,-140.0


## 4. Train / test split — per cell

In [8]:
X_COLUMNS = [c.CELL_EL_DEG, c.LOG_DISTANCE, c.RELATIVE_BEARING, c.ANTENNA_GAIN]
Y_COLUMNS = ["avg_rsrp"]
TEST_SIZE = 0.30

# BDT/ExactGP is capped separately because it does not scale to the full data.
MAX_TRAIN_PER_CELL_BDT = 15000

# SVGP uses all available rows after the train/test split. Leave as None for full data.
MAX_TRAIN_PER_CELL_SVGP = None
MAX_TEST_PER_CELL = None


def maybe_sample(df: pd.DataFrame, max_rows: int | None, seed: int = 42) -> pd.DataFrame:
    if max_rows is None or len(df) <= max_rows:
        return df
    return df.sample(n=max_rows, random_state=seed)

train_map_bdt, train_map_svgp, test_map = {}, {}, {}
for cell_id, df in ue_data_df.groupby(c.CELL_ID):
    train, test = train_test_split(df, test_size=TEST_SIZE, random_state=42)

    train_bdt = maybe_sample(train, MAX_TRAIN_PER_CELL_BDT)
    train_svgp = maybe_sample(train, MAX_TRAIN_PER_CELL_SVGP)
    test = maybe_sample(test, MAX_TEST_PER_CELL)

    train_map_bdt[cell_id] = train_bdt.reset_index(drop=True)
    train_map_svgp[cell_id] = train_svgp.reset_index(drop=True)
    test_map[cell_id] = test.reset_index(drop=True)

cell_ids = sorted(test_map)
for cell_id in cell_ids:
    print(
        f"{cell_id}: "
        f"BDT train={len(train_map_bdt[cell_id]):>6,}, "
        f"SVGP train={len(train_map_svgp[cell_id]):>6,}, "
        f"test={len(test_map[cell_id]):>6,}"
    )

print("\nTotals:")
print(f"BDT train rows  : {sum(len(v) for v in train_map_bdt.values()):,}")
print(f"SVGP train rows : {sum(len(v) for v in train_map_svgp.values()):,}")
print(f"Test rows       : {sum(len(v) for v in test_map.values()):,}")

train_list_bdt = [train_map_bdt[cid] for cid in cell_ids]
train_list_svgp = [train_map_svgp[cid] for cid in cell_ids]
test_list = [test_map[cid] for cid in cell_ids]


cell_1_0: BDT train=15,000, SVGP train=35,000, test=15,000
cell_1_1: BDT train=15,000, SVGP train=35,000, test=15,000
cell_1_2: BDT train=15,000, SVGP train=35,000, test=15,000
cell_1_3: BDT train=15,000, SVGP train=35,000, test=15,000
cell_2_0: BDT train=15,000, SVGP train=35,000, test=15,000
cell_2_1: BDT train=15,000, SVGP train=35,000, test=15,000
cell_2_2: BDT train=15,000, SVGP train=35,000, test=15,000
cell_2_3: BDT train=15,000, SVGP train=35,000, test=15,000
cell_3_0: BDT train=15,000, SVGP train=35,000, test=15,000
cell_3_1: BDT train=15,000, SVGP train=35,000, test=15,000
cell_3_2: BDT train=15,000, SVGP train=35,000, test=15,000
cell_3_3: BDT train=15,000, SVGP train=35,000, test=15,000
cell_4_0: BDT train=15,000, SVGP train=35,000, test=15,000
cell_4_1: BDT train=15,000, SVGP train=35,000, test=15,000
cell_4_2: BDT train=15,000, SVGP train=35,000, test=15,000
cell_4_3: BDT train=15,000, SVGP train=35,000, test=15,000
cell_5_0: BDT train=15,000, SVGP train=35,000, test=15,0

In [9]:
x_max = {
    c.CELL_EL_DEG: 50,
    c.CELL_LAT: 90,
    c.CELL_LON: 180,
    c.LOG_DISTANCE: 12,
    c.RELATIVE_BEARING: 360,
    c.ANTENNA_GAIN: 40,
}
x_min = {
    c.CELL_EL_DEG: -10,
    c.CELL_LAT: -90,
    c.CELL_LON: -180,
    c.LOG_DISTANCE: 0,
    c.RELATIVE_BEARING: 0,
    c.ANTENNA_GAIN: -40,
}


## 5. Train BayesianDTModel (ExactGP)

In [10]:
bayesian_model_map = {}
bayesian_losses = {}
bayesian_train_time = 0.0
RUN_BAYESIAN = True
if RUN_BAYESIAN:
    if BayesianDigitalTwin is None:
        raise ImportError("BayesianDigitalTwin is unavailable in this checkout")

    t0 = time.time()
    for cell_id in cell_ids:
        train_df = train_map_bdt[cell_id]
        model = BayesianDigitalTwin([train_df], X_COLUMNS, Y_COLUMNS, x_max=x_max, x_min=x_min)
        loss = model.train_distributed_gpmodel(
            maxiter=35,
            lr=0.05,
            stopping_threshold=1e-4,
        )
        bayesian_model_map[cell_id] = model
        bayesian_losses[cell_id] = loss[loss != 0]
        print(
            f"  {cell_id}: {len(train_df):>5} pts, {len(bayesian_losses[cell_id])} iters, "
            f"final loss = {bayesian_losses[cell_id][-1]:.4f}"
        )

    bayesian_train_time = time.time() - t0
    print(f"\nBayesian total time: {bayesian_train_time:.2f}s")
else:
    print("Skipping Bayesian ExactGP. Set RUN_BAYESIAN = True for small comparison runs.")


[2026-04-28 00:16:14,669] INFO:  Iter 1/35 - Loss: 0.774 (delta=inf)
[2026-04-28 00:16:17,241] INFO:  Iter 2/35 - Loss: 0.756 (delta=-0.017663)
[2026-04-28 00:16:19,465] INFO:  Iter 3/35 - Loss: 0.738 (delta=-0.018303)
[2026-04-28 00:16:21,080] INFO:  Iter 4/35 - Loss: 0.720 (delta=-0.017858)
[2026-04-28 00:16:22,615] INFO:  Iter 5/35 - Loss: 0.702 (delta=-0.017940)
[2026-04-28 00:16:25,293] INFO:  Iter 6/35 - Loss: 0.684 (delta=-0.018237)
[2026-04-28 00:16:27,078] INFO:  Iter 7/35 - Loss: 0.665 (delta=-0.018569)
[2026-04-28 00:16:28,808] INFO:  Iter 8/35 - Loss: 0.647 (delta=-0.018264)
[2026-04-28 00:16:31,645] INFO:  Iter 9/35 - Loss: 0.629 (delta=-0.018326)
[2026-04-28 00:16:33,301] INFO:  Iter 10/35 - Loss: 0.610 (delta=-0.018891)
[2026-04-28 00:16:34,782] INFO:  Iter 11/35 - Loss: 0.592 (delta=-0.018251)
[2026-04-28 00:16:36,375] INFO:  Iter 12/35 - Loss: 0.572 (delta=-0.019169)
[2026-04-28 00:16:37,887] INFO:  Iter 13/35 - Loss: 0.554 (delta=-0.018859)
[2026-04-28 00:16:39,401] I

  cell_1_0: 15000 pts, 35 iters, final loss = 0.1323


[2026-04-28 00:17:15,493] INFO:  Iter 1/35 - Loss: 0.776 (delta=inf)
[2026-04-28 00:17:17,027] INFO:  Iter 2/35 - Loss: 0.758 (delta=-0.017493)
[2026-04-28 00:17:18,560] INFO:  Iter 3/35 - Loss: 0.741 (delta=-0.017496)
[2026-04-28 00:17:21,201] INFO:  Iter 4/35 - Loss: 0.723 (delta=-0.017692)
[2026-04-28 00:17:22,829] INFO:  Iter 5/35 - Loss: 0.705 (delta=-0.017793)
[2026-04-28 00:17:24,323] INFO:  Iter 6/35 - Loss: 0.687 (delta=-0.017762)
[2026-04-28 00:17:25,843] INFO:  Iter 7/35 - Loss: 0.669 (delta=-0.018076)
[2026-04-28 00:17:27,341] INFO:  Iter 8/35 - Loss: 0.651 (delta=-0.018161)
[2026-04-28 00:17:28,819] INFO:  Iter 9/35 - Loss: 0.633 (delta=-0.018198)
[2026-04-28 00:17:30,371] INFO:  Iter 10/35 - Loss: 0.615 (delta=-0.018215)
[2026-04-28 00:17:33,170] INFO:  Iter 11/35 - Loss: 0.596 (delta=-0.018348)
[2026-04-28 00:17:34,687] INFO:  Iter 12/35 - Loss: 0.578 (delta=-0.018520)
[2026-04-28 00:17:36,248] INFO:  Iter 13/35 - Loss: 0.559 (delta=-0.018717)
[2026-04-28 00:17:37,834] I

  cell_1_1: 15000 pts, 35 iters, final loss = 0.1481


[2026-04-28 00:18:12,108] INFO:  Iter 1/35 - Loss: 0.765 (delta=inf)
[2026-04-28 00:18:13,619] INFO:  Iter 2/35 - Loss: 0.748 (delta=-0.017690)
[2026-04-28 00:18:15,124] INFO:  Iter 3/35 - Loss: 0.730 (delta=-0.017887)
[2026-04-28 00:18:16,622] INFO:  Iter 4/35 - Loss: 0.712 (delta=-0.017948)
[2026-04-28 00:18:18,145] INFO:  Iter 5/35 - Loss: 0.694 (delta=-0.018219)
[2026-04-28 00:18:19,688] INFO:  Iter 6/35 - Loss: 0.675 (delta=-0.018179)
[2026-04-28 00:18:21,202] INFO:  Iter 7/35 - Loss: 0.657 (delta=-0.018481)
[2026-04-28 00:18:22,788] INFO:  Iter 8/35 - Loss: 0.638 (delta=-0.018700)
[2026-04-28 00:18:24,306] INFO:  Iter 9/35 - Loss: 0.620 (delta=-0.018500)
[2026-04-28 00:18:25,832] INFO:  Iter 10/35 - Loss: 0.601 (delta=-0.018849)
[2026-04-28 00:18:27,351] INFO:  Iter 11/35 - Loss: 0.582 (delta=-0.018915)
[2026-04-28 00:18:28,930] INFO:  Iter 12/35 - Loss: 0.563 (delta=-0.019122)
[2026-04-28 00:18:30,511] INFO:  Iter 13/35 - Loss: 0.544 (delta=-0.019161)
[2026-04-28 00:18:32,163] I

...

  cell_5_1: 15000 pts, 35 iters, final loss = 0.0954


[2026-04-28 00:33:50,378] INFO:  Iter 1/35 - Loss: 0.766 (delta=inf)
[2026-04-28 00:33:51,895] INFO:  Iter 2/35 - Loss: 0.748 (delta=-0.018222)
[2026-04-28 00:33:53,512] INFO:  Iter 3/35 - Loss: 0.729 (delta=-0.018891)
[2026-04-28 00:33:55,098] INFO:  Iter 4/35 - Loss: 0.710 (delta=-0.018648)
[2026-04-28 00:33:56,966] INFO:  Iter 5/35 - Loss: 0.692 (delta=-0.018626)
[2026-04-28 00:33:58,525] INFO:  Iter 6/35 - Loss: 0.673 (delta=-0.018754)
[2026-04-28 00:34:00,813] INFO:  Iter 7/35 - Loss: 0.654 (delta=-0.019001)
[2026-04-28 00:34:02,404] INFO:  Iter 8/35 - Loss: 0.635 (delta=-0.018904)
[2026-04-28 00:34:03,957] INFO:  Iter 9/35 - Loss: 0.616 (delta=-0.018493)
[2026-04-28 00:34:05,537] INFO:  Iter 10/35 - Loss: 0.597 (delta=-0.019637)
[2026-04-28 00:34:07,086] INFO:  Iter 11/35 - Loss: 0.578 (delta=-0.019117)
[2026-04-28 00:34:08,659] INFO:  Iter 12/35 - Loss: 0.558 (delta=-0.019543)
[2026-04-28 00:34:10,196] INFO:  Iter 13/35 - Loss: 0.539 (delta=-0.018981)
[2026-04-28 00:34:12,063] I

  cell_5_2: 15000 pts, 35 iters, final loss = 0.0902


[2026-04-28 00:34:47,610] INFO:  Iter 1/35 - Loss: 0.768 (delta=inf)
[2026-04-28 00:34:49,138] INFO:  Iter 2/35 - Loss: 0.750 (delta=-0.018262)
[2026-04-28 00:34:50,675] INFO:  Iter 3/35 - Loss: 0.732 (delta=-0.018355)
[2026-04-28 00:34:52,213] INFO:  Iter 4/35 - Loss: 0.713 (delta=-0.018452)
[2026-04-28 00:34:53,762] INFO:  Iter 5/35 - Loss: 0.695 (delta=-0.018445)
[2026-04-28 00:34:55,320] INFO:  Iter 6/35 - Loss: 0.677 (delta=-0.018187)
[2026-04-28 00:34:57,574] INFO:  Iter 7/35 - Loss: 0.658 (delta=-0.018700)
[2026-04-28 00:34:59,190] INFO:  Iter 8/35 - Loss: 0.639 (delta=-0.018869)
[2026-04-28 00:35:01,626] INFO:  Iter 9/35 - Loss: 0.620 (delta=-0.018667)
[2026-04-28 00:35:03,180] INFO:  Iter 10/35 - Loss: 0.602 (delta=-0.018821)
[2026-04-28 00:35:04,707] INFO:  Iter 11/35 - Loss: 0.583 (delta=-0.019076)
[2026-04-28 00:35:06,210] INFO:  Iter 12/35 - Loss: 0.564 (delta=-0.018984)
[2026-04-28 00:35:07,800] INFO:  Iter 13/35 - Loss: 0.544 (delta=-0.019280)
[2026-04-28 00:35:09,346] I

  cell_5_3: 15000 pts, 35 iters, final loss = 0.1092

Bayesian total time: 1175.07s


## 6. Train SVGPDigitalTwin

In [ ]:
min_train_svgp = min(len(df) for df in train_list_svgp)
svgp_config = SVGPTrainConfig(
    num_inducing=min(1000, min_train_svgp),
    batch_size=2048,
    num_epochs=100,
    learning_rate=0.01,
    ngd_lr=0.1,
    stopping_threshold=1e-4,
    inducing_init="kmeans++",
    seed=42,
    log_every=1,
    multicell_batched=True,  # batched multi-cell SVGP via IndependentMultitaskVariationalStrategy
)
mode_str = "multicell batched (MultiCellSVGPModel)" if svgp_config.multicell_batched else "per-cell sequential"
print(
    f"SVGP config: {svgp_config.num_inducing} inducing pts, "
    f"batch={svgp_config.batch_size}, epochs={svgp_config.num_epochs}, "
    f"init={svgp_config.inducing_init}"
)
print(f"SVGP mode   : {mode_str}")
print(f"SVGP full train rows: {sum(len(df) for df in train_list_svgp):,}")

svgp_model = SVGPDigitalTwin(x_max=x_max, x_min=x_min, device="cpu")

t0 = time.time()
svgp_loss_matrix = svgp_model.train(train_list_svgp, X_COLUMNS, Y_COLUMNS, svgp_config)
svgp_train_time = time.time() - t0

svgp_cell_ids = list(svgp_model._cell_ids)
svgp_losses = {
    cid: svgp_loss_matrix[i][np.isfinite(svgp_loss_matrix[i])]
    for i, cid in enumerate(svgp_cell_ids)
}

for cid in svgp_cell_ids:
    print(
        f"  {cid}: {len(train_map_svgp[cid]):>6,} pts, {len(svgp_losses[cid])} epochs, "
        f"final loss = {svgp_losses[cid][-1]:.4f}"
    )
print(f"\nSVGP total time: {svgp_train_time:.2f}s")


def predict_bayesian_cell(cell_id: str, frame: pd.DataFrame):
    if not RUN_BAYESIAN:
        n = len(frame)
        return np.full(n, np.nan), np.full(n, np.nan)
    means, stds = bayesian_model_map[cell_id].predict_distributed_gpmodel([frame.copy()])
    return means[0], stds[0]


## 7. Training loss curves

In [13]:
cell_ids = sorted(test_map.keys())
n_cells = len(cell_ids)
fig, axes = plt.subplots(1, n_cells, figsize=(5 * n_cells, 4))
if n_cells == 1:
    axes = [axes]

for ax, cid in zip(axes, cell_ids):
    if RUN_BAYESIAN:
        ax.plot(bayesian_losses[cid], "b-o", markersize=3, label="Bayesian (Exact MLL)")
    ax.plot(svgp_losses[cid], "r-s", markersize=3, label="SVGP (Variational ELBO)")
    ax.set_title(cid)
    ax.set_xlabel("Iter / Epoch")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Training Loss Curves", fontsize=13)
plt.tight_layout()
plt.show()


## 8. Prediction — MAE and MAPE

In [14]:
results = []
pred_cache = {}

# Full SVGP test evaluation in one call. With the full Million dataset this is
# 20 frames x 15,000 rows = 300,000 held-out predictions.
ordered_test_frames = [test_map[cid].copy() for cid in svgp_cell_ids]
svgp_test_means, svgp_test_stds = svgp_model.predict(ordered_test_frames)

for idx, cell_id in enumerate(svgp_cell_ids):
    true_rsrp = test_map[cell_id]["avg_rsrp"].values
    svgp_m = svgp_test_means[:, idx]
    svgp_s = svgp_test_stds[:, idx]
    bay_m, bay_s = predict_bayesian_cell(cell_id, test_map[cell_id])
    pred_cache[cell_id] = {
        "bay_m": bay_m,
        "bay_s": bay_s,
        "svgp_m": svgp_m,
        "svgp_s": svgp_s,
    }

    row = {
        "cell_id": cell_id,
        "n_train_bdt": len(train_map_bdt[cell_id]),
        "n_train_svgp": len(train_map_svgp[cell_id]),
        "n_test": len(test_map[cell_id]),
        "svgp_mae_db": round(float(np.abs(true_rsrp - svgp_m).mean()), 3),
        "svgp_mape_%": round(float((100 * np.abs((true_rsrp - svgp_m) / true_rsrp)).mean()), 2),
        "svgp_uncertainty_db": round(float(svgp_s.mean()), 3),
    }
    if RUN_BAYESIAN:
        row.update({
            "bayesian_mae_db": round(float(np.abs(true_rsrp - bay_m).mean()), 3),
            "bayesian_mape_%": round(float((100 * np.abs((true_rsrp - bay_m) / true_rsrp)).mean()), 2),
            "bayesian_uncertainty_db": round(float(bay_s.mean()), 3),
        })
    results.append(row)

results_df = pd.DataFrame(results).set_index("cell_id")
results_df


,n_train_bdt,n_train_svgp,n_test,svgp_mae_db,svgp_mape_%,svgp_uncertainty_db,bayesian_mae_db,bayesian_mape_%,bayesian_uncertainty_db
cell_id,,,,,,,,,
cell_1_0,15000,35000,15000,1.857,2.49,5.267,1.832,2.46,11.016
cell_1_1,15000,35000,15000,1.783,2.42,5.434,1.809,2.41,10.669
cell_1_2,15000,35000,15000,1.572,2.17,4.669,1.733,2.24,11.101
cell_1_3,15000,35000,15000,2.135,2.88,5.339,1.875,2.51,10.930
cell_2_0,15000,35000,15000,1.325,1.69,3.752,1.250,1.63,8.975
cell_2_1,15000,35000,15000,1.385,1.76,3.716,1.276,1.66,9.044
cell_2_2,15000,35000,15000,1.324,1.69,3.694,1.359,1.69,8.989
cell_2_3,15000,35000,15000,1.285,1.61,3.539,1.362,1.66,9.047
cell_3_0,15000,35000,15000,1.983,2.58,5.429,2.077,2.65,10.484


In [ ]:
print("=== Summary ===")
if RUN_BAYESIAN:
    print(f"Bayesian avg MAE : {results_df['bayesian_mae_db'].mean():.3f} dB")
print(f"SVGP     avg MAE : {results_df['svgp_mae_db'].mean():.3f} dB")
if RUN_BAYESIAN:
    print(f"Bayesian time    : {bayesian_train_time:.2f}s")
mode_str = "multicell batched (MultiCellSVGPModel)" if svgp_config.multicell_batched else "per-cell sequential"
print(f"SVGP time        : {svgp_train_time:.2f}s")
print(f"SVGP mode        : {mode_str}")
print(f"Num cells        : {len(svgp_cell_ids)}")
results_df


## 9. Side-by-side scatter: True vs Predicted RSRP

In [ ]:
cell_ids = sorted(test_map.keys())
n_cells = len(cell_ids)
fig, axes = plt.subplots(1, n_cells, figsize=(5 * n_cells, 4))
if n_cells == 1:
    axes = [axes]

for ax, cid in zip(axes, cell_ids):
    true_r = test_map[cid]["avg_rsrp"].values
    sm = pred_cache[cid]["svgp_m"]
    series = [true_r, sm]
    labels = ["SVGP"]
    if RUN_BAYESIAN:
        bm = pred_cache[cid]["bay_m"]
        series.append(bm)
        labels.insert(0, "Bayesian")
    lim = [min(arr.min() for arr in series) - 2, max(arr.max() for arr in series) + 2]
    ax.plot(lim, lim, "k--", lw=1, alpha=0.5, label="Perfect")
    if RUN_BAYESIAN:
        ax.scatter(true_r, pred_cache[cid]["bay_m"], alpha=0.5, s=10, c="steelblue", label="Bayesian")
    ax.scatter(true_r, sm, alpha=0.5, s=10, c="tomato", marker="s", label="SVGP")
    r = results_df.loc[cid]
    title = f"{cid}\nSVGP={r['svgp_mae_db']}dB MAE"
    if RUN_BAYESIAN:
        title = f"{cid}\nBay={r['bayesian_mae_db']}dB | SVGP={r['svgp_mae_db']}dB MAE"
    ax.set_title(title)
    ax.set_xlabel("True RSRP (dBm)")
    ax.set_ylabel("Predicted (dBm)")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(lim)
    ax.set_ylim(lim)

fig.suptitle("True vs Predicted RSRP", fontsize=13)
plt.tight_layout()
plt.show()


: 

## 10. Uncertainty calibration — ±2σ bands vs log_distance

In [ ]:
cell_ids = sorted(test_map.keys())
n_cells = len(cell_ids)
fig, axes = plt.subplots(1, n_cells, figsize=(5 * n_cells, 4))
if n_cells == 1:
    axes = [axes]

for ax, cid in zip(axes, cell_ids):
    tdf = test_map[cid].copy()
    sort_idx = tdf[c.LOG_DISTANCE].argsort().values
    tdf_sorted = tdf.iloc[sort_idx].reset_index(drop=True)

    # Reuse predictions already computed in cell-21 — no extra predict calls needed.
    sm = pred_cache[cid]["svgp_m"][sort_idx]
    ss = pred_cache[cid]["svgp_s"][sort_idx]

    x = tdf_sorted[c.LOG_DISTANCE].values
    ax.scatter(x, tdf_sorted["avg_rsrp"].values, c="black", s=6, zorder=5, label="Measured")
    if RUN_BAYESIAN:
        bm = pred_cache[cid]["bay_m"][sort_idx]
        bs = pred_cache[cid]["bay_s"][sort_idx]
        ax.plot(x, bm, "b-", lw=1.5, label="Bayesian")
        ax.fill_between(x, bm - 2 * bs, bm + 2 * bs, alpha=0.15, color="blue")
    ax.plot(x, sm, "r--", lw=1.5, label="SVGP")
    ax.fill_between(x, sm - 2 * ss, sm + 2 * ss, alpha=0.15, color="red")
    ax.set_xlabel("log_distance")
    ax.set_ylabel("RSRP (dBm)")
    ax.set_title(f"{cid} - +/-2 sigma")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

fig.suptitle("Prediction Uncertainty", fontsize=13)
plt.tight_layout()
plt.show()


## 11. RSRP coverage heatmap (SVGP)

In [ ]:
GRID_N = 40
grid_lats = np.linspace(LAT_MIN, LAT_MAX, GRID_N)
grid_lons = np.linspace(LON_MIN, LON_MAX, GRID_N)
LON_GRID, LAT_GRID = np.meshgrid(grid_lons, grid_lats)

grid_template = pd.DataFrame({c.LOC_X: LON_GRID.ravel(), c.LOC_Y: LAT_GRID.ravel()})
prediction_frames = SVGPDigitalTwin.create_prediction_frames(site_configs_demo, grid_template)
ordered_grid_frames = [prediction_frames[cid] for cid in svgp_cell_ids]
means, _ = svgp_model.predict(ordered_grid_frames)

best_rsrp = means.max(axis=1)
rsrp_map = best_rsrp.reshape(GRID_N, GRID_N)
print(f"Predicted RSRP: {rsrp_map.min():.1f} to {rsrp_map.max():.1f} dBm")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.pcolormesh(LON_GRID, LAT_GRID, rsrp_map,
                   cmap="RdYlGn", vmin=-120, vmax=-50, shading="auto")
plt.colorbar(im, ax=ax, label="Best-cell RSRP (dBm)")

colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12"]
for (_, row), color in zip(site_configs_demo.iterrows(), colors):
    ax.scatter(row.cell_lon, row.cell_lat, s=250, marker="^",
               c=color, edgecolors="black", linewidths=1.5, zorder=10, label=row.cell_id)

ax.scatter(ue_data_df["lon"], ue_data_df["lat"],
           c=ue_data_df["avg_rsrp"], cmap="RdYlGn", vmin=-120, vmax=-50,
           s=5, alpha=0.4, edgecolors="none", zorder=5)

ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title(f"SVGP Digital Twin — Best-Cell RSRP Coverage\n"
             f"{DEMO_SITE} ({len(demo_cell_ids)} cells, Mobility Data)")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout(); plt.show()

## 12. Save / load model (torch.save, no pickle)

In [ ]:
import os
import tempfile

with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as f:
    path = f.name

svgp_model.save(path)
file_kb = os.path.getsize(path) / 1024
print(f"Saved SVGP model: {file_kb:.1f} KB -> {path}")

loaded = SVGPDigitalTwin.load(path, map_location="cpu")
multicell_mode = getattr(loaded, "_multicell_batched", False)
print(
    f"Loaded: is_trained={loaded.is_trained}, "
    f"cell_ids={loaded._cell_ids}, "
    f"multicell_batched={multicell_mode}"
)

check_frames = [test_map[cid].head(50).copy() for cid in svgp_cell_ids]
orig_m, _ = svgp_model.predict([frame.copy() for frame in check_frames])
load_m, _ = loaded.predict([frame.copy() for frame in check_frames])
diff = np.abs(orig_m - load_m).max()
print(f"Max prediction diff post-load: {diff:.6f} dB  {'OK' if diff < 0.01 else 'DIFF LARGE'}")
os.unlink(path)


## 14. When does SVGP outperform Bayesian? — Scaling analysis

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║              SVGP vs Bayesian: Accuracy & Scalability                   ║
╠══════════════╦══════════════════════╦══════════════════════════════════╣
║ n pts/cell   ║ Bayesian (ExactGP)   ║ SVGP (m=500 inducing)            ║
╠══════════════╬══════════════════════╬══════════════════════════════════╣
║ 50           ║ BEST accuracy, fast  ║ Good — slight variance overhead  ║
║ 500          ║ Good, 0.1-2s/cell    ║ Good — on par within ~2-5% MAE  ║
║ 5,000        ║ ~60s/cell (slow!)    ║ ~5s/cell — 12x faster           ║
║ 50,000       ║ OOM / hours          ║ ~50s/cell — still feasible      ║
║ 500,000      ║ Impossible           ║ ~8min/cell — works               ║
╚══════════════╩══════════════════════╩══════════════════════════════════╝

Key insight:
  • SVGP is a variational APPROXIMATION to ExactGP.
  • On small n (like this notebook), Bayesian is marginally more accurate.
  • SVGP's real advantage: it CAN RUN on real deployment data where Bayesian
    completely fails (O(n³) Cholesky → OOM or hours).
  • With m ≈ 500 inducing points and n ≈ 5K-50K/cell, SVGP MAE is within
    2-5% of ExactGP — practically indistinguishable in production.
  • Both models use the same kernel family, so the accuracy ceiling is the
    same; SVGP just reaches it with O(n·m²) instead of O(n³).
""")

## 15. DTModel interface summary

In [ ]:
print(f"SVGPDigitalTwin is DTModel: {isinstance(svgp_model, DTModel)}")
print(f"SVGP model_type: {svgp_model.model_type}")
print()
print("Current SVGP interface:")
print("  model.train(data_in, x_columns, y_columns, config)")
print("  model.predict(prediction_dfs) -> (means, stds)")
print("  model.save(path)              -> torch.save checkpoint")
print("  SVGPDigitalTwin.load(path)    -> restored model")
print("  model.is_trained              -> bool")
print("  model.model_type              -> str")
